# SHEAR v3: SmolVLA fine-tune on a free T4 (Colab or Kaggle)

Trains SmolVLA (`lerobot/smolvla_base`) on the SHEAR v1-rig dataset (base camera + claw camera, scripted cuts): the **action expert and the vision encoder** train, the language layers stay frozen. Same recipe as `policy/train_smolvla_lora.sh v3`, moved to a 16 GB T4. The same notebook runs on Google Colab and on Kaggle; cell 1 finds out which.

**Before you run, on Colab**
1. Runtime → Change runtime type → **T4 GPU**.
2. **Secrets** (key icon on the left): add `HF_TOKEN` = a Hugging Face token with *write* access, and switch on "Notebook access". The dataset and the checkpoints are private repos on the Hub.
3. Run all. Keep the tab open: the free tier ends a session after a while, and stops giving a GPU for a day or so after a few hours of use.

**Before you run, on Kaggle** (about 30 GPU hours a week, at most 12 h per run)
1. The account needs a verified phone number (Settings), or the GPU and the internet stay switched off.
2. Notebook sidebar: Session options → Accelerator **GPU T4 x2** (one is used), Internet **on**.
3. Add-ons → **Secrets**: add `HF_TOKEN` (same token as above) and tick it for this notebook.
4. **Save Version → Save & Run All**. That run goes on with the browser closed; its log is on the version's page.

**If the session dies or hits its time limit**, run all again (on Kaggle: save a new version). The train cell finds the last checkpoint on the Hub and resumes from it (same step counter, same optimizer state, same data order). Nothing is lost except the steps since the last checkpoint (at most `SAVE_FREQ`). Never run it in two places at once: both would write the same checkpoints.

In [ ]:
# 1. Where this runs (Colab or Kaggle), the GPU it got, the Python, and the free disk.
import os, sys, shutil, subprocess
import torch
KAGGLE = os.path.exists("/kaggle")
print("platform:", "Kaggle" if KAGGLE else "Colab", "| python", sys.version.split()[0])
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
# lerobot falls back to the CPU without a word when there is no GPU, and would crawl for weeks: stop here instead.
assert torch.cuda.is_available(), "no GPU in this session: " + (
    "Settings -> Accelerator -> GPU T4 x2 (needs a phone-verified Kaggle account)" if KAGGLE
    else "Runtime -> Change runtime type -> T4 GPU") + ", then run all again"
print("GPU ok:", torch.cuda.get_device_name(0))

# Training files (dataset cache aside): the smoke run and the local checkpoints, ~2.8 GB each, the older ones deleted
# once they are on the Hub (cell 6). Kaggle's /kaggle/working is capped at 20 GB and saved as the run's output, so the
# training files go to /tmp there; only the logs go to /kaggle/working, where they are kept with the run.
WORK = "/tmp/shear" if KAGGLE else "/content"
LOGS = "/kaggle/working" if KAGGLE else "/content"
os.makedirs(WORK, exist_ok=True)
for p in sorted({WORK, LOGS, os.path.expanduser("~")}):
    print(f"free disk at {p}: {shutil.disk_usage(p).free / 1e9:.0f} GB")


In [ ]:
# 2. lerobot, pinned to the SAME commit as the laptop's fork (0.6.1 + 70, 2026-11), with the SmolVLA and video extras.
#    It needs Python 3.12 or newer. If this notebook's Python has it (Colab does), lerobot goes straight into it.
#    If not, lerobot goes into a separate Python 3.12 in /tmp/lr312, and only the training runs there (it is a
#    separate process anyway). A few minutes either way. If a later cell fails on an import, restart the session and
#    run all again.
LEROBOT = "lerobot[smolvla,av-dep] @ git+https://github.com/huggingface/lerobot@5aa74557"
if sys.version_info >= (3, 12):
    %pip install -q "{LEROBOT}"
    PY = sys.executable
    TRAIN = shutil.which("lerobot-train") or "lerobot-train"
else:
    %pip install -q uv
    !uv venv -q --python 3.12 /tmp/lr312
    # torch from PyTorch's CUDA 12.8 builds, which run on any driver from CUDA 12.0 up (PyPI's default may need newer)
    !uv pip install -q --python /tmp/lr312/bin/python "torch>=2.7,<2.12" "torchvision>=0.22,<0.27" --index-url https://download.pytorch.org/whl/cu128
    !uv pip install -q --python /tmp/lr312/bin/python "{LEROBOT}"
    PY, TRAIN = "/tmp/lr312/bin/python", "/tmp/lr312/bin/lerobot-train"
r = subprocess.run([PY, "-c", "import lerobot, torch; print('lerobot', lerobot.__version__, 'torch', torch.__version__, "
                    "'cuda', torch.cuda.is_available())"], capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0 and "cuda True" in r.stdout, "lerobot is missing, or the training Python sees no GPU: read the lines above"


In [ ]:
# 3. Patch lerobot's SmolVLA, in the Python that trains (cell 2). The same code as policy/patch_lerobot_smolvla.py:
#    written to a file here, then run with that Python.
PATCH = r'''# The SHEAR edits to lerobot's SmolVLA (the same as the laptop's local lerobot, K7 2026-11-06).
#    a) Upstream, train_expert_only freezes the WHOLE vision-language model, vision encoder included, so
#       --policy.freeze_vision_encoder=false did nothing. Now the vision encoder trains with the action expert and
#       the language layers stay frozen (186M of 450M params train).
#    b) SmolVLA loads the vision-language model and the expert in bfloat16. The trainable weights are kept in
#       float32 instead (fp16 still does the fast compute on the T4): the fp16 gradient scaler can't handle
#       bfloat16 gradients ("not implemented for 'BFloat16'"), and small updates get lost in bfloat16 weights.
#    Safe to run again: each edit checks whether it is already there.
import pathlib
import lerobot.policies.smolvla.smolvlm_with_expert as m

PATH = pathlib.Path(m.__file__)
s = PATH.read_text()

def put(s, anchor, new, where):
    n = s.count(anchor)
    assert n == 1, f"anchor found {n} times, expected once: the lerobot version changed"
    return s.replace(anchor, new + anchor if where == "before" else anchor + new)

edits = []
if "if self.train_expert_only and not self.freeze_vision_encoder:" not in s:
    s = put(s, "        # To avoid unused params issue with distributed training\n"
               "        for name, params in self.lm_expert.named_parameters():\n",
            "        # SHEAR: the vision encoder trains with the expert\n"
            "        if self.train_expert_only and not self.freeze_vision_encoder:\n"
            "            self.get_vlm_model().vision_model.train()\n"
            "            for params in self.get_vlm_model().vision_model.parameters():\n"
            "                params.requires_grad = True\n", "before")
    edits.append("vision encoder trainable")
if "vision_model.train(mode)" not in s:
    s = put(s, "            self.get_vlm_model().vision_model.eval()\n\n"
               "        if self.train_expert_only:\n            self.vlm.eval()\n",
            "            if not self.freeze_vision_encoder:\n"
            "                self.get_vlm_model().vision_model.train(mode)\n", "after")
    edits.append("vision encoder in train mode")
if "SHEAR fp32 master weights" not in s:
    s = put(s, "        for name, params in self.lm_expert.named_parameters():\n"
               "            if \"lm_head\" in name:\n"
               "                params.requires_grad = False\n",
            "        # SHEAR fp32 master weights: trainable weights in float32, the frozen ones stay bfloat16\n"
            "        for params in self.parameters():\n"
            "            if params.requires_grad and params.dtype == torch.bfloat16:\n"
            "                params.data = params.data.float()\n", "after")
    edits.append("trainable weights in float32")
if "SHEAR connector dtype" not in s:
    s = put(s, "        # Modality projection & resampling\n",
            "        # SHEAR connector dtype: the float32 vision encoder feeds the frozen bfloat16 connector\n"
            "        image_hidden_states = image_hidden_states.to(\n"
            "            dtype=next(self.get_vlm_model().connector.parameters()).dtype\n"
            "        )\n", "before")
    edits.append("connector input cast")
compile(s, str(PATH), "exec")
PATH.write_text(s)
print("patched:" if edits else "already patched:", ", ".join(edits) or "all four edits present", "->", PATH)
'''
open("/tmp/patch_lerobot_smolvla.py", "w").write(PATCH)
r = subprocess.run([PY, "/tmp/patch_lerobot_smolvla.py"], capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:])
assert r.returncode == 0, "the patch failed: read the lines above"

In [ ]:
# 4. Hugging Face login + the run's settings.
import os
from huggingface_hub import whoami

# The token comes from the platform's secret store (see the top of the notebook), never from the notebook itself.
if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
else:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("logged in as", whoami()["name"])

DATASET = "krish5831/shear_sim_scripted_v1"      # 200 scripted episodes on the v1 rig (private)
MODEL_REPO = "krish5831/shear_smolvla_v3"         # every checkpoint lands here under checkpoints/<step>/ (private)
# SmolVLA's base expects its own camera names. The base camera is the overview, the claw camera the wrist view.
# The SAME map goes to every rollout of the trained policy (sim/rollout_policy.py does this).
RENAME = '{"observation.images.base": "observation.images.camera1", "observation.images.claw": "observation.images.camera2"}'
STEPS = 100_000        # K7's call. lerobot counts every batch as a step: 100k x 8 = 800k samples, ~5.6 passes over the data.
BATCH, ACCUM = 8, 4    # 8 per batch on the T4, gradients summed over 4 batches = effective batch 32, as on the laptop
SAVE_FREQ = 5_000      # a checkpoint on the Hub every 5k steps: what a dead session costs at most
PRECISION = "fp16"     # fast fp16 compute on the T4 (no bf16 hardware); the trained weights stay float32 (cell 3).
                       # If the smoke prints loss nan, set "no" (slower) and run cells 4 and 5 again.

COMMON = [
    f"--policy.path=lerobot/smolvla_base", f"--dataset.repo_id={DATASET}", "--dataset.video_backend=pyav",
    "--policy.device=cuda", f"--rename_map={RENAME}", "--wandb.enable=false",
    "--policy.freeze_vision_encoder=false", f"--batch_size={BATCH}", f"--accelerator.gradient_accumulation.steps={ACCUM}",
    "--num_workers=2",
]
# "no" is lerobot's default, and its config parser reads a literal no as the boolean False: pass the flag only for fp16.
if PRECISION != "no":
    COMMON.append(f"--accelerator.mixed_precision={PRECISION}")

# Runs lerobot-train and shows its log in this cell as it comes (a subprocess's own output never reaches the page),
# also writing it to a log file. The stop button ends the training too: otherwise it keeps running unseen and holds
# the GPU. The progress bar updates every 30 s instead of every 0.1 s, so hours of training don't flood the page.
import subprocess

def run_live(cmd, log_path):
    env = dict(os.environ, TQDM_MININTERVAL="30", PYTHONUNBUFFERED="1")
    lines = []
    pr = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    try:
        with open(log_path, "a") as f:
            for line in pr.stdout:
                if "INFO" in line and not line.startswith("INFO"):
                    line = line.replace("INFO", "\nINFO", 1)   # a progress-bar update glued to a log line
                print(line, end="", flush=True); f.write(line); f.flush(); lines.append(line)
        pr.wait()
    except BaseException:
        pr.terminate()
        try:
            pr.wait(timeout=60)
        except subprocess.TimeoutExpired:
            pr.kill()
        print("\n-- stopped; the training process is ended too")
        raise
    if pr.returncode != 0:
        raise RuntimeError(f"lerobot-train stopped with an error (exit {pr.returncode}): read the lines above, also in {log_path}")
    return lines

In [ ]:
# 5. Smoke: 30 steps. Downloads the dataset (3.3 GB) and the base model, checks the loss is a number,
#    and prints the time per step -> the hours the full run needs. Nothing goes to the Hub from this cell.
#    If it fails with "not implemented for 'BFloat16'", cell 3 is an old version: open the notebook again from
#    GitHub (the dev branch) and run all again.
#    If it fails with "CUDA out of memory", set BATCH, ACCUM = 4, 8 in cell 4 (same effective batch) and rerun 4 and 5.
import re, shutil

shutil.rmtree(f"{WORK}/smoke", ignore_errors=True)
cmd = [TRAIN, *COMMON, "--steps=30", "--save_freq=30", "--log_freq=10",
       "--policy.push_to_hub=false", f"--output_dir={WORK}/smoke", "--job_name=smoke"]
log = run_live(cmd, f"{LOGS}/smoke.log")
steps = [l for l in log if "updt_s:" in l]
loss = re.search(r"loss:([\d.na]+)", steps[-1]).group(1)
updt = float(re.search(r"updt_s:([\d.]+)", steps[-1]).group(1))
data = float(re.search(r"data_s:([\d.]+)", steps[-1]).group(1))
print(f"\nloss {loss}   {updt:.2f} s per step + {data:.2f} s loading   ->   {STEPS} steps = {STEPS*(updt+data)/3600:.1f} h")
if loss == "nan":
    print("!! loss is nan: set PRECISION = 'no' in cell 4 and run cells 4 and 5 again")

In [ ]:
# 6. The run. Fresh if the Hub repo holds no checkpoint yet, otherwise RESUMED from the newest one.
#    Every SAVE_FREQ steps a checkpoint goes to MODEL_REPO (weights 0.9 GB + optimizer state ~1.5 GB).
#    Run this cell again after any disconnect. A log line every 50 steps (the first after a few minutes),
#    a progress line with the time left every 30 s. The same lines go to train_v3_<time>.log in LOGS (cell 1).
#    Local checkpoints older than the newest two are deleted (they are on the Hub by then): each is ~2.8 GB.
import datetime as dt, pathlib, shutil, threading, time
from huggingface_hub import HfApi

api = HfApi()
try:
    have = [f for f in api.list_repo_files(MODEL_REPO) if f.startswith("checkpoints/")]
except Exception:
    have = []
stamp = dt.datetime.now().strftime("%m%d_%H%M%S")
OUT = f"{WORK}/train/v3_{stamp}"
if have:
    newest = max(f.split("/")[1] for f in have)
    print("resuming from", MODEL_REPO, "checkpoint", newest)
    cmd = [TRAIN, "--resume=true", f"--config_path={MODEL_REPO}", f"--output_dir={OUT}"]
else:
    print("fresh run ->", MODEL_REPO)
    cmd = [TRAIN, *COMMON, f"--steps={STEPS}", f"--save_freq={SAVE_FREQ}", "--log_freq=50",
           "--save_checkpoint_to_hub=true", "--policy.push_to_hub=true", f"--policy.repo_id={MODEL_REPO}",
           "--policy.private=true", f"--output_dir={OUT}", "--job_name=shear_smolvla_v3"]
def prune(out, keep=2):
    # lerobot uploads a checkpoint before it goes on training, so all but the newest are already on the Hub
    while True:
        time.sleep(600)
        done = sorted(p for p in pathlib.Path(out, "checkpoints").glob("[0-9]*") if p.is_dir())
        for p in done[:-keep]:
            shutil.rmtree(p, ignore_errors=True)

threading.Thread(target=prune, args=(OUT,), daemon=True).start()
run_live(cmd, f"{LOGS}/train_v3_{stamp}.log")

In [ ]:
# 7. What is on the Hub now.
from huggingface_hub import HfApi
files = HfApi().list_repo_files(MODEL_REPO)
print(sorted({f.split("/")[1] for f in files if f.startswith("checkpoints/")}))

## Picking a checkpoint

The loss says little about cuts (Dream Machines' π0.5 write-up: pick checkpoints by rollouts, not loss). Each checkpoint is rolled out in the sim on the laptop, on rows the policy never saw:

```bash
# on the laptop, lerobot env, S.H.E.A.R root
hf download krish5831/shear_smolvla_v3 --include "checkpoints/020000/pretrained_model/*" --local-dir outputs/train/shear_smolvla_v3_hub
python sim/rollout_policy.py --ckpt outputs/train/shear_smolvla_v3_hub/checkpoints/020000/pretrained_model --rtc 10 --rtc-horizon 20 --smooth 0.5 --windowed
```

Stop the run when two checkpoints in a row cut no better than the one before.